In [10]:
# Load the original dataset to start the full transformation process
original_file_path =original_file_path = r'C:\Users\Admin\Downloads\Side Projects\Utah Hockey Club Contest\BDC_2024_Womens_Data.csv'

original_data = pd.read_csv(original_file_path)

# Define relevant events to include in the possession sequence
possession_events = ['Play', 'Puck Recovery', 'Faceoff Win', 'Takeaway']

# Define the full transformation process as a function
def full_data_transformation(data):
    import networkx as nx

    # Step 1: Identify sequences that finish with a shot on net or a goal
    def identify_sequences_with_shot_or_goal(data):
        sequences = []  # List to store all identified possession sequences
        current_sequence = []  # List to store the current sequence being constructed
        possession_team = None  # Track the team that is in possession

        current_period = None  # Track the current period
        current_game = None  # Track the current game (combination of Date, Home Team, and Away Team)
        sequence_id = 0  # Track the sequence ID

        all_sequences_with_ids = []  # Store all sequences with sequence and play IDs

        for index, row in data.iterrows():
            event_team = row['Team']
            event_type = row['Event']
            period = row['Period']
            game_id = (row['Date'], row['Home Team'], row['Away Team'])  # Use Date, Home Team, and Away Team as game ID

            # Reset sequence when the period or game changes
            if period != current_period or game_id != current_game:
                current_sequence = []  # Reset the sequence
                possession_team = None  # Reset possession
                current_period = period  # Update to the new period
                current_game = game_id  # Update to the new game

            # Start a new sequence if a faceoff win occurs (even if the team is the same)
            if event_type == 'Faceoff Win':
                current_sequence = []  # Start a new sequence
                possession_team = event_team  # Update the possession team

            # If the event is a possession event and the team gains possession (reset the sequence for the shot/goal-taking team)
            if event_type in possession_events and event_team != possession_team:
                current_sequence = []  # Start a new sequence
                possession_team = event_team  # Update the possession team

            # Add the event to the sequence if the possession is with the team that will eventually take the shot or score the goal
            if event_type in possession_events and event_team == possession_team:
                row_copy = row.copy()  # Copy row to avoid SettingWithCopyWarning
                current_sequence.append(row_copy)

            # When the event is a Shot on Net or Goal and the current team is in possession, finalize and store the sequence
            if (event_type == 'Shot' and row['Detail 2'] == 'On Net') or event_type == 'Goal':
                row_copy = row.copy()  # Copy row to avoid SettingWithCopyWarning
                current_sequence.append(row_copy)  # Add the shot or goal event
                if current_sequence:  # Only store sequences that end with a shot on net or goal
                    for i, play in enumerate(current_sequence):
                        play['Sequence ID'] = sequence_id
                        play['Play ID'] = i + 1
                        all_sequences_with_ids.append(play)
                    sequence_id += 1  # Increment sequence ID
                current_sequence = []  # Reset the sequence for the next shot or goal
                possession_team = None  # Reset possession

        return pd.DataFrame(all_sequences_with_ids)  # Return the dataframe with sequences and IDs

    # Step 2: Perform flow centrality calculations
    def calculate_flow_centrality(sequences_df):
        flow_centrality_results = []
        weight_decay = 0.8  # Decay factor

        # Iterate through each sequence
        for sequence_id in sequences_df['Sequence ID'].unique():
            sequence = sequences_df[sequences_df['Sequence ID'] == sequence_id]
            G_shot = nx.DiGraph()

            previous_player = None
            shot_player = None
            shot_nodes = set()

            # Create a directed graph for the sequence
            for _, event in sequence.iterrows():
                current_player = event['Player']
                event_type = event['Event']

                # Add edge between players in sequence
                if previous_player is not None:
                    G_shot.add_edge(previous_player, current_player)

                # Handle the shot event
                if event_type == 'Shot' and event['Detail 2'] == 'On Net':
                    G_shot.add_edge(current_player, 'Shot')  # Shot node
                    shot_player = current_player
                    shot_nodes.add('Shot')

                elif event_type == 'Goal':
                    G_shot.add_edge(current_player, 'Goal')  # Goal node
                    shot_player = current_player
                    shot_nodes.add('Goal')

                previous_player = current_player

            # Calculate flow centrality
            flow_centrality = {}
            if shot_player:
                # Assign the player who took the shot a centrality value of 1
                flow_centrality[shot_player] = 1.0

                for node in G_shot.nodes:
                    if node in shot_nodes:
                        continue

                    flow_centrality[node] = 0  # Initialize

                    for shot_node in shot_nodes:
                        try:
                            paths = list(nx.all_shortest_paths(G_shot, source=node, target=shot_node))
                            for path in paths:
                                path_length = len(path) - 1  # Number of steps to the shot
                                for i, player in enumerate(path[:-1]):  # Exclude the shot node
                                    weight = weight_decay ** (path_length - i)
                                    if player not in flow_centrality:
                                        flow_centrality[player] = 0
                                    flow_centrality[player] += weight
                        except nx.NetworkXNoPath:
                            continue

            # Store flow centrality results for this sequence
            for player, centrality in flow_centrality.items():
                flow_centrality_results.append({
                    'Sequence ID': sequence_id,
                    'Player': player,
                    'Flow Centrality': centrality
                })

        return pd.DataFrame(flow_centrality_results)

    # Step 3: Merge flow centrality results onto sequences
    sequences_with_shot_or_goal = identify_sequences_with_shot_or_goal(data)
    flow_centrality_df = calculate_flow_centrality(sequences_with_shot_or_goal)
    merged_data = sequences_with_shot_or_goal.merge(flow_centrality_df, on=['Sequence ID', 'Player'], how='left')

    # Step 4: Add 'X2' and 'Y2' columns for coordinates of the next play in the sequence
    merged_data['X2'] = merged_data['X Coordinate'].shift(-1)
    merged_data['Y2'] = merged_data['Y Coordinate'].shift(-1)
    merged_data.loc[merged_data['Sequence ID'] != merged_data['Sequence ID'].shift(-1), ['X2', 'Y2']] = None

    return merged_data

# Perform the full transformation
final_transformed_data = full_data_transformation(original_data)

# Save the final dataset with flow centrality and coordinates to a CSV file
final_output_file = r'C:\Users\Admin\Downloads\Side Projects\Utah Hockey Club Contest\FinalSubmission(2).csv'

final_transformed_data.to_csv(final_output_file, index=False)

final_output_file  # Returning the file path to access the output


'C:\\Users\\Admin\\Downloads\\Side Projects\\Utah Hockey Club Contest\\FinalSubmission(2).csv'

In [11]:
import pandas as pd 